## 6 Независимая итоговая оценка

**Входы**

- Зафиксированные Pipeline и порог.
- Test, не использованный при обучении и подборе.

**Выходы**

- Метрики, матрица ошибок, ROC/PR-кривые и диагностика калибровки.
- Приближённые интервалы неопределённости метрик.

**Что делаем**

Применяем готовое решение без fit и изменения порога. Положительный класс 1 — дефолт. Сравниваем с Dummy на той же выборке.

In [ ]:
# Только для разработки: открыть раздел можно из notebooks/ или корня проекта
from pathlib import Path as _NotebookPath
import os as _notebook_os
_notebook_root = next(
    path for path in (_NotebookPath.cwd(), *_NotebookPath.cwd().parents)
    if (path / 'config/build/project.json').is_file()
)
_notebook_os.chdir(_notebook_root)


Сообщаем AP и ROC-AUC для ранжирования, precision/recall/F1/F2 для выбранного порога, accuracy и balanced accuracy, Brier для вероятностного прогноза. Матрица ошибок содержит абсолютные количества.

Для оценки неопределённости выполняем **300 стратифицированных bootstrap-перевыборок test** с random seed 42. В каждой сохраняем число объектов каждого класса, используем уже готовые прогнозы и тот же порог; приводим процентили 2.5% и 97.5%. Интервалы условны относительно этого разбиения и обученной модели, не учитывают переобучение, неизвестную зависимость клиентов и изменение доли дефолтов.

График калибровки — диагностика, а не новый этап настройки: никакой калибратор по test не обучается.

In [ ]:
exec(compile(__import__('pathlib').Path('src/notebooks/_code/evaluate.py').read_text(encoding='utf-8'), 'src/notebooks/_code/evaluate.py', 'exec'), globals())

#### Ограничения и применимость

Наблюдаемые ошибки характеризуют выбранное случайное разбиение датасета Kaggle. Источник не даёт достаточных сведений о сроке наблюдения дефолта, времени заявок, идентификаторах заёмщиков и репрезентативности выборки. Редкие грейды и малые группы ограничивают надёжность частных выводов. Bootstrap не устраняет эти ограничения.

Грейд и ставка должны быть доступны в момент прогноза; для задачи до скоринга кредитора набор признаков нужно пересмотреть и провести новый эксперимент. Случайное разбиение не заменяет временную и внешнюю валидацию. Не оценены финансовый эффект, стабильность и справедливость решений по группам. Итоговые показатели сами по себе не дают основания внедрять автоматический отказ в кредитовании.